# Weak test sources: which ones, and why

Follow-up to `Per_Source_Rank_Performance.ipynb`. That notebook used a 20-source sample; this one ranks **all test
sources** for the 256 px model (`cbm_final_res256`, step 300k) and looks for the cause of each weak source.

**Lens: kingdom accuracy.** The rank analysis showed that 78–90% of errors at every lower rank are first made at
kingdom, so a source that is weak at kingdom is weak everywhere (the mean accuracy over all six ranks is shown
alongside to confirm this).

Sections:
1. **Ranking:** per-source kingdom accuracy; the bottom `WEAK_QUANTILE` are the "weak" sources.
2. **Label mix vs source effect:** each source's *expected* accuracy if each of its labels scored as it does in the
   other test sources. Low expected = the source is full of labels that are hard everywhere; actual ≪ expected = this
   source specifically goes wrong.
3. **Composition and error attribution:** what the weak sources contain (label groups from the concept table) and
   which groups their errors come from.
4. **Training coverage:** are the labels rare or absent in the training data?
5. **Prediction bias:** does the model shift a whole source towards one kingdom?
6. **Image properties:** colour, brightness, sharpness, aspect ratio, resolution vs training images; and whether the
   768 px model does better.
7. **Examples:** images from the weakest sources with the predictions drawn on.
8. **Diagnosis table and interpretation.**

In [ ]:
import json
import sys
from multiprocessing import Pool
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from PIL import Image

REPO = Path("../..").resolve()
sys.path.insert(0, str(REPO))

DUMP = REPO / "results_tmp/cbm_final_res256_step300000/test"
DUMP_768 = REPO / "results_tmp/cbm_cd275000_res768_step50000/test"   # optional comparison (taxv2 table!)
TRAIN_COUNTS = REPO / "dataset_stats/train_label_counts_final_768.parquet"  # same CoralNet train sources as final_256
TRAIN_CONFIG = Path("/data/vision/beery/scratch/sauder/mermaid-segmentation-final/configs/data_config_final.yaml")
IMG_ROOT = Path("/archive/vision/beery/red_sea_coral/cache_MERMAID/dev-datamermaid-sm-sources")
MANIFEST = IMG_ROOT / "etl-outputs/coralnet/20260623_nogit/coralnet_training_manifest_20260623_nogit.parquet"
CACHE_DIR = REPO / "results_tmp/source_diagnosis"   # image statistics are cached here

SEED = 0
MIN_SCORED = 200       # min kingdom-labelled points for a source to be ranked
WEAK_QUANTILE = 0.25   # bottom share of sources (by kingdom accuracy) called "weak"
LOSO_MIN = 50          # min points of a label in *other* sources to use its accuracy there as the expectation
RARE_TRAIN = 1_000     # a label with fewer CoralNet training points than this counts as rare
IMG_PER_TEST_SOURCE, IMG_PER_TRAIN_SOURCE = 20, 5
N_EXAMPLE_SOURCES, N_EXAMPLE_IMAGES = 12, 3
FLAG = dict(dead_coral=0.20, other_algae=0.30, rare=0.25, resid=-0.10, bias=0.30, unscored=0.50)

RANKS = ["kingdom", "phylum", "class", "order", "family", "genus"]
INK = "#0b0b0b"; INK2 = "#52514e"; GRID = "#e4e3df"; SURFACE = "#fcfcfb"
BLUE, ORANGE, GREEN = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": GRID,
                     "axes.labelcolor": INK2, "xtick.color": INK2, "ytick.color": INK2, "text.color": INK,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 10})
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 200)

cols = (["source_id", "image_id", "image_key", "row", "col", "source_label_name", "gt__live", "gt__algae"]
        + [f"{p}_{r}" for r in RANKS for p in ("gt", "pred")])
df = pd.read_parquet(DUMP / "points.parquet", columns=cols)
for c in df.columns:
    if df[c].dtype.name == "category":
        df[c] = df[c].astype(object)
images = pd.read_parquet(DUMP / "images.parquet")
meta = json.loads((DUMP / "meta.json").read_text())
df["label"] = df.source_label_name.astype(str).str.strip().str.lower()
print(meta["checkpoint"].split("/")[-2:], f"{len(df):,} points, {df.source_id.nunique()} sources, "
      f"{len(images):,} images")

In [ ]:
# Label groups, from each point's concept row (same definitions as the kingdom section of the rank notebook).
GROUPS = ["dead_coral", "live_hardcoral", "other_animal", "turf", "other_algae", "abiotic", "other_kingdom"]
GROUP_COLORS = {"dead_coral": "#e87ba4", "live_hardcoral": "#eb6834", "other_animal": "#eda100",
                "turf": "#1baf7a", "other_algae": "#008300", "abiotic": "#b8b6b0", "other_kingdom": "#2a78d6"}
k = df.gt_kingdom
df["group"] = np.select(
    [k.isna(), (k == "animalia") & (df.gt__live == 1), (k == "animalia") & (df.gt_order == "scleractinia"),
     k == "animalia", (k == "none") & (df.gt__algae == 2), k.isin(["plantae", "chromista"]), k == "none"],
    ["unscored", "dead_coral", "live_hardcoral", "other_animal", "turf", "other_algae", "abiotic"], "other_kingdom")
df["ok"] = (df.gt_kingdom == df.pred_kingdom).where(k.notna())

tr = pd.read_parquet(TRAIN_COUNTS)
train_n = tr[tr.dataset == "coralnet"].groupby("label").n.sum()
df["train_n"] = df.label.map(train_n).fillna(0)

sc = df[df.gt_kingdom.notna()]   # points scored at kingdom
print(f"{len(sc):,} of {len(df):,} points have a kingdom label")
sc.groupby("group").ok.agg(points="size", kingdom_accuracy="mean").sort_values("points", ascending=False).round(3)

## 1. Ranking the test sources

One row per source. `acc_<rank>`: accuracy at that rank (`none` included, as in the eval); `acc_mean`: mean over the
six ranks; `living_kingdom`: kingdom accuracy without `none`-annotated points. Sources with fewer than `MIN_SCORED`
kingdom-labelled points are not ranked (listed below the figure).

In [ ]:
rows = []
for s, g in df.groupby("source_id"):
    r = {"source_id": s, "points": len(g), "images": g.image_id.nunique(), "scored": int(g.gt_kingdom.notna().sum())}
    for rank in RANKS:
        x = g[g[f"gt_{rank}"].notna()]
        r[f"acc_{rank}"] = (x[f"gt_{rank}"] == x[f"pred_{rank}"]).mean() if len(x) else np.nan
    liv = g[g.gt_kingdom.notna() & (g.gt_kingdom != "none")]
    r["living_kingdom"] = (liv.gt_kingdom == liv.pred_kingdom).mean() if len(liv) >= 30 else np.nan
    rows.append(r)
S = pd.DataFrame(rows).set_index("source_id")
S["acc_mean"] = S[[f"acc_{r}" for r in RANKS]].mean(axis=1)
S["unscored"] = 1 - S.scored / S.points

ranked = S[S.scored >= MIN_SCORED].sort_values("acc_kingdom")
cut = ranked.acc_kingdom.quantile(WEAK_QUANTILE)
WEAK = ranked[ranked.acc_kingdom <= cut].index.tolist()
S["weak"] = S.index.isin(WEAK)
print(f"{len(ranked)} ranked sources; weak = kingdom accuracy ≤ {cut:.3f} ({len(WEAK)} sources)")
print("not ranked (too few kingdom-labelled points):", S[S.scored < MIN_SCORED].index.tolist())
print("Spearman(kingdom acc, mean acc over ranks) =", round(ranked.acc_kingdom.corr(ranked.acc_mean, method="spearman"), 2))

fig, ax = plt.subplots(figsize=(15, 4.2))
x = np.arange(len(ranked))
ax.axvspan(-0.5, len(WEAK) - 0.5, color=GRID, alpha=0.6, lw=0)
ax.scatter(x, ranked.acc_kingdom, s=10 + 2 * np.sqrt(ranked.points), color=BLUE, alpha=0.8, label="kingdom accuracy", zorder=3)
ax.scatter(x, ranked.acc_mean, s=14, marker="_", color=INK2, label="mean accuracy over 6 ranks", zorder=3)
ax.scatter(x, ranked.living_kingdom, s=12, marker="x", color=ORANGE, label="kingdom accuracy, `none` removed", zorder=3)
ax.set_xticks(x[: len(WEAK)], ranked.index[: len(WEAK)], rotation=90, fontsize=7.5)
ax.set_xlim(-1, len(ranked)); ax.set_ylim(0, 1.02); ax.grid(axis="y", color=GRID)
ax.set_xlabel("test sources, sorted by kingdom accuracy (ids shown for the weak ones, shaded)")
ax.set_ylabel("accuracy")
ax.set_title(f"Per-source accuracy, all {len(ranked)} ranked test sources (marker area ~ √points)", loc="left")
ax.legend(frameon=False, loc="lower right")
fig.tight_layout(); plt.show()
ranked.loc[WEAK, ["points", "images", "scored", "acc_kingdom", "living_kingdom", "acc_mean", "acc_genus"]].round(2)

## 2. Label mix vs source-specific effect

For every (source, label), the **expected** kingdom accuracy is that label's accuracy in all *other* test sources
(leave-source-out; labels with fewer than `LOSO_MIN` points elsewhere fall back to their label group's accuracy). A
source's expected accuracy is the point-weighted mean over its labels.

- **Low expected accuracy**: the source is full of labels that are hard everywhere (label-mix problem; it will be the
  same for any source with that mix).
- **Residual = actual − expected ≪ 0**: labels that work elsewhere fail in this source (images, annotation
  convention, habitat).

In [ ]:
lab_tot = sc.groupby("label").ok.agg(["sum", "size"])
ls = sc.groupby(["source_id", "label"]).ok.agg(["sum", "size"]).reset_index().join(lab_tot, on="label", rsuffix="_all")
ls["n_else"] = ls.size_all - ls["size"]
ls["acc_here"] = ls["sum"] / ls["size"]
ls["acc_else"] = np.where(ls.n_else >= LOSO_MIN, (ls.sum_all - ls["sum"]) / ls.n_else.clip(lower=1), np.nan)
label_group = sc.groupby("label").group.agg(lambda s: s.mode().iloc[0])
group_acc = sc.groupby("group").ok.mean()
ls["expected"] = ls.acc_else.fillna(ls.label.map(label_group).map(group_acc).astype(float))
ls["unique_here"] = ls.n_else < LOSO_MIN
e = ls.groupby("source_id").apply(lambda g: pd.Series({
    "expected": np.average(g.expected, weights=g["size"]),
    "unique_label_share": np.average(g.unique_here, weights=g["size"])}), include_groups=False)
S = S.join(e)
S["residual"] = S.acc_kingdom - S.expected
r = S.loc[ranked.index]
print(f"r(expected, actual) = {r.expected.corr(r.acc_kingdom):.2f} → the label mix explains "
      f"{r.expected.corr(r.acc_kingdom) ** 2:.0%} of the between-source variance; the rest is source-specific.")

fig, ax = plt.subplots(figsize=(8.5, 7.5))
w = r[r.weak]; o = r[~r.weak]
ax.plot([0, 1], [0, 1], color=INK2, lw=1, ls="--", zorder=1)
ax.fill_between([0, 1], [0 + FLAG["resid"], 1 + FLAG["resid"]], [-1, 0], color=ORANGE, alpha=0.07, lw=0)
ax.scatter(o.expected, o.acc_kingdom, s=10 + 2 * np.sqrt(o.points), color=GRID, edgecolors=INK2, lw=0.6, label="other sources")
ax.scatter(w.expected, w.acc_kingdom, s=10 + 2 * np.sqrt(w.points), color=BLUE, alpha=0.85, label="weak sources")
for s, q in w.iterrows():
    ax.annotate(str(s), (q.expected, q.acc_kingdom), xytext=(4, 2), textcoords="offset points", fontsize=8)
ax.set_xlim(0.25, 1); ax.set_ylim(0.25, 1); ax.set_aspect("equal"); ax.grid(color=GRID)
ax.set_xlabel("expected kingdom accuracy from the label mix (labels scored as in other sources)")
ax.set_ylabel("actual kingdom accuracy")
ax.set_title("Below the diagonal: worse than its labels predict.\nShaded: residual < "
             f"{FLAG['resid']} (source-specific problem)", loc="left", fontsize=10)
ax.legend(frameon=False, loc="upper left")
fig.tight_layout(); plt.show()
S.loc[WEAK, ["acc_kingdom", "expected", "residual", "unique_label_share"]].sort_values("residual").round(2)

## 3. What the weak sources contain, and where their errors come from

Left: share of each weak source's kingdom-labelled points in each label group. Right: share of its kingdom **errors**
from each group. Groups come from the concept table: `dead_coral` = animalia with `live` = False; `turf` = kingdom
`none` with `algae` = True; `other_algae` = plantae / chromista; `abiotic` = other `none` (sand, rock, rubble,
shadow, tape...).

In [ ]:
order = ranked.loc[WEAK].index
comp = pd.crosstab(sc.source_id, sc.group, normalize="index").reindex(index=order, columns=GROUPS, fill_value=0)
errs = sc[sc.ok == 0]
err_share = pd.crosstab(errs.source_id, errs.group, normalize="index").reindex(index=order, columns=GROUPS, fill_value=0)
S = S.join(pd.crosstab(sc.source_id, sc.group, normalize="index").add_prefix("share_"))

fig, axes = plt.subplots(1, 2, figsize=(14, 0.28 * len(order) + 1.8), sharey=True)
for ax, tab, title in [(axes[0], comp, "share of kingdom-labelled points"), (axes[1], err_share, "share of kingdom errors")]:
    left = np.zeros(len(tab))
    for gname in GROUPS:
        ax.barh(range(len(tab)), tab[gname], left=left, color=GROUP_COLORS[gname], label=gname, height=0.8)
        left += tab[gname].to_numpy()
    ax.set_xlim(0, 1); ax.set_title(title, loc="left"); ax.grid(axis="x", color=GRID)
axes[0].set_yticks(range(len(order)), [f"{s}  ({ranked.acc_kingdom[s]:.2f})" for s in order], fontsize=8)
axes[0].invert_yaxis(); axes[0].set_ylabel("weak source  (kingdom accuracy)")
axes[1].legend(frameon=False, bbox_to_anchor=(1.01, 1), loc="upper left")
fig.tight_layout(); plt.show()

## 4. Training coverage of the test labels

Kingdom accuracy by how many CoralNet training points carry the same raw label (all points, pooled), and for each
weak source the share of its points in labels that are rare (< `RARE_TRAIN`) or absent in training. The model
predicts concepts, not raw labels, so an unseen label can still be right when its concepts are common; this measures
how unfamiliar a source's vocabulary (and usually its organisms) is.

In [ ]:
bins = [-1, 0, 100, 1_000, 10_000, 100_000, np.inf]
names = ["0", "1–100", "100–1k", "1k–10k", "10k–100k", ">100k"]
cov = sc.groupby(pd.cut(sc.train_n, bins, labels=names), observed=False).ok.agg(points="size", accuracy="mean")
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(cov.index.astype(str), cov.accuracy, color=BLUE)
for i, (n, a) in enumerate(zip(cov.points, cov.accuracy)):
    ax.text(i, a + 0.01, f"{a:.2f}\n{n:,} pts", ha="center", va="bottom", fontsize=8)
ax.set_ylim(0, 1); ax.set_xlabel("CoralNet training points with the same raw label"); ax.set_ylabel("kingdom accuracy")
ax.set_title("Accuracy rises with training coverage of the label", loc="left"); ax.grid(axis="y", color=GRID)
fig.tight_layout(); plt.show()

S["absent_share"] = sc.assign(v=sc.train_n == 0).groupby("source_id").v.mean()
S["rare_share"] = sc.assign(v=sc.train_n < RARE_TRAIN).groupby("source_id").v.mean()
print("Spearman with kingdom accuracy over ranked sources:",
      S.loc[ranked.index, ["rare_share", "absent_share"]].corrwith(S.loc[ranked.index, "acc_kingdom"], method="spearman").round(2).to_dict())
S.loc[WEAK, ["acc_kingdom", "rare_share", "absent_share", "unique_label_share"]].sort_values("rare_share", ascending=False).round(2)

## 5. Does the model shift a whole source towards one kingdom?

For each source: the annotated vs predicted kingdom distribution. `shift` = total-variation distance between them
(0 = predicted mix matches the annotated mix; 1 = completely different). `shift_to` = the kingdom that is most
over-predicted. A large shift means the model reads the whole source differently (e.g. everything looks like
`animalia`), which points at the images or the source's annotation convention rather than individual labels.

In [ ]:
gt_mix = pd.crosstab(sc.source_id, sc.gt_kingdom, normalize="index")
pr_mix = pd.crosstab(sc.source_id, sc.pred_kingdom, normalize="index")
allk = gt_mix.columns.union(pr_mix.columns)
gt_mix, pr_mix = gt_mix.reindex(columns=allk, fill_value=0), pr_mix.reindex(columns=allk, fill_value=0)
S["shift"] = 0.5 * (pr_mix - gt_mix).abs().sum(axis=1)
S["shift_to"] = (pr_mix - gt_mix).idxmax(axis=1)
S["shift_by"] = (pr_mix - gt_mix).max(axis=1)
r = S.loc[ranked.index]
print("Spearman(shift, kingdom accuracy) =", round(r["shift"].corr(r.acc_kingdom, method="spearman"), 2),
      "| Spearman(shift, residual) =", round(r["shift"].corr(r.residual, method="spearman"), 2))

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(r["shift"], r.acc_kingdom, s=10 + 2 * np.sqrt(r.points), c=np.where(r.weak, BLUE, GRID), edgecolors=INK2, lw=0.5)
for s, q in r[r.weak & (r["shift"] >= FLAG["bias"])].iterrows():
    ax.annotate(f"{s} → {q.shift_to}", (q["shift"], q.acc_kingdom), xytext=(4, 2), textcoords="offset points", fontsize=8)
ax.set_xlabel("shift: distance between annotated and predicted kingdom mix"); ax.set_ylabel("kingdom accuracy")
ax.set_title("Weak sources (blue) are those whose predicted kingdom mix is far from the annotated one", loc="left", fontsize=10)
ax.grid(color=GRID); fig.tight_layout(); plt.show()

## 6. Image properties

Per-image statistics on a sample of `IMG_PER_TEST_SOURCE` images per test source and `IMG_PER_TRAIN_SOURCE` per
training source (thumbnails ≤ 512 px): brightness, contrast, saturation, colour fractions, sharpness (log Laplacian
variance), share of very dark / clipped pixels, aspect ratio and megapixels. `img_dist` = standardised distance from
a test source's mean statistics to the *nearest* training source. Cached in `CACHE_DIR`.

The 768 px model column is a cheap resolution test, but that model uses the taxv2 concept table, so part of any
difference comes from different ground truth for a few labels.

In [ ]:
def image_stats(key: str) -> dict | None:
    try:
        im = Image.open(IMG_ROOT / key).convert("RGB")
        w, h = im.size
        im.thumbnail((512, 512))
        a = np.asarray(im, dtype=np.float32) / 255
    except Exception:
        return None
    g = a.mean(axis=2); hsv = np.asarray(im.convert("HSV"), dtype=np.float32) / 255; tot = a.sum(axis=2) + 1e-6
    lap = 4 * g[1:-1, 1:-1] - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    return dict(key=key, bright=g.mean(), contrast=g.std(), sat=hsv[..., 1].mean(), blue_frac=(a[..., 2] / tot).mean(),
                green_frac=(a[..., 1] / tot).mean(), red_frac=(a[..., 0] / tot).mean(),
                sharp=float(np.log10(lap.var() + 1e-9)), dark=(g < 0.1).mean(), clipped=(g > 0.97).mean(),
                aspect=max(w, h) / min(w, h), mpix=w * h / 1e6)

def cached_stats(name: str, keys: pd.DataFrame) -> pd.DataFrame:
    path = CACHE_DIR / f"image_stats_{name}.parquet"
    if path.exists():
        return pd.read_parquet(path)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    with Pool(32) as pool:
        out = [r for r in pool.imap(image_stats, keys.key.tolist(), chunksize=4) if r]
    res = pd.DataFrame(out).merge(keys, on="key")
    res.to_parquet(path)
    return res

test_keys = (images.groupby("source_id", group_keys=False)
             .apply(lambda g: g.sample(min(len(g), IMG_PER_TEST_SOURCE), random_state=SEED), include_groups=False)
             .rename(columns={"image_key": "key"}))
test_keys["source_id"] = images.loc[test_keys.index, "source_id"]
train_sources = set(yaml.safe_load(TRAIN_CONFIG.read_text())["data"]["coralnet"]["train"]["whitelist_sources"])
man = pd.read_parquet(MANIFEST, columns=["source_id", "image_id", "image_s3_key"])
man = man[man.source_id.isin(train_sources)].drop_duplicates("image_id")
train_keys = (man.groupby("source_id", group_keys=False)
              .apply(lambda g: g.sample(min(len(g), IMG_PER_TRAIN_SOURCE), random_state=SEED), include_groups=False)
              .rename(columns={"image_s3_key": "key"}))
train_keys["source_id"] = man.loc[train_keys.index, "source_id"]
ist_test = cached_stats("test", test_keys[["key", "source_id"]])
ist_train = cached_stats("train", train_keys[["key", "source_id"]])
print(f"image stats: {len(ist_test):,} test images, {len(ist_train):,} train images ({ist_train.source_id.nunique()} train sources)")

F = ["bright", "contrast", "sat", "blue_frac", "green_frac", "red_frac", "sharp", "dark", "clipped"]
tr_src = ist_train.groupby("source_id")[F + ["aspect", "mpix"]].median()
te_src = ist_test.groupby("source_id")[F + ["aspect", "mpix"]].median()
mu, sd = tr_src[F].mean(), tr_src[F].std()
zt, zr = (te_src[F] - mu) / sd, (tr_src[F] - mu) / sd
te_src["img_dist"] = [np.sqrt(((zr.to_numpy() - zt.loc[s].to_numpy()) ** 2).sum(axis=1)).min() for s in zt.index]
S = S.drop(columns=[c for c in te_src.columns if c in S.columns]).join(te_src)

if (DUMP_768 / "points.parquet").exists():
    q = pd.read_parquet(DUMP_768 / "points.parquet", columns=["source_id", "gt_kingdom", "pred_kingdom"])
    q = q[q.gt_kingdom.notna()]
    S["acc_kingdom_768"] = (q.gt_kingdom.astype(str) == q.pred_kingdom.astype(str)).groupby(q.source_id).mean()
    S["gain_768"] = S.acc_kingdom_768 - S.acc_kingdom

r = S.loc[ranked.index]
print("Training sources: aspect ratio 99th pct", round(tr_src.aspect.quantile(0.99), 2), "| megapixels 5th pct",
      round(tr_src.mpix.quantile(0.05), 2), "| img_dist of train sources to their nearest other train source is not computed;",
      "test median img_dist", round(r.img_dist.median(), 2))
corr = r[F + ["aspect", "mpix", "img_dist"]].corrwith(r.residual, method="spearman").rename("vs residual").to_frame()
corr["vs kingdom acc"] = r[F + ["aspect", "mpix", "img_dist"]].corrwith(r.acc_kingdom, method="spearman")
display(corr.round(2).T)
outl = S.loc[WEAK][(S.loc[WEAK].aspect > tr_src.aspect.quantile(0.99)) | (S.loc[WEAK].mpix < tr_src.mpix.quantile(0.05))
                   | (S.loc[WEAK].img_dist > r.img_dist.quantile(0.9))]
print("weak sources whose images are unusual (aspect, resolution or colour/brightness far from every training source):")
display(outl[["acc_kingdom", "residual", "aspect", "mpix", "img_dist", "bright", "sat", "green_frac", "blue_frac"]].round(2))
if "gain_768" in S:
    print(f"768 px vs 256 px kingdom accuracy: median gain over ranked sources {r.gain_768.median():+.3f}; weak sources:")
    display(S.loc[WEAK, ["acc_kingdom", "acc_kingdom_768", "gain_768"]].sort_values("gain_768", ascending=False).round(2).T)

## 7. Example images from the weakest sources

`N_EXAMPLE_IMAGES` random images from each of the `N_EXAMPLE_SOURCES` weakest sources. Point fill = annotated
kingdom; a thick black edge marks a wrong kingdom prediction, and the first few wrong points are labelled
`raw label → predicted kingdom`.

In [ ]:
KCOL = {"animalia": "#e8463c", "plantae": "#2fbf4a", "chromista": "#c79a2c", "none": "#e0e0e0", "bacillati": "#2a78d6"}

def show_source(s: int, n: int = N_EXAMPLE_IMAGES, axes=None):
    x = sc[sc.source_id == s]
    ids = x.image_id.drop_duplicates().sample(min(n, x.image_id.nunique()), random_state=SEED)
    for ax, iid in zip(axes, ids):
        y = x[x.image_id == iid]
        im = Image.open(IMG_ROOT / y.image_key.iloc[0]).convert("RGB"); w0 = im.size[0]; im.thumbnail((900, 900)); f = im.size[0] / w0
        ok = (y.gt_kingdom == y.pred_kingdom).to_numpy()
        ax.imshow(im)
        ax.scatter(y.col * f, y.row * f, s=26, c=[KCOL.get(v, "k") for v in y.gt_kingdom],
                   edgecolors=np.where(ok, "white", "black"), linewidths=np.where(ok, 0.8, 2.2))
        for _, p in y[~ok].head(4).iterrows():
            ax.text(p.col * f + 6, p.row * f, f"{p.label[:20]} → {p.pred_kingdom}", fontsize=6.5, color="yellow",
                    bbox=dict(fc="black", alpha=0.55, pad=0.5, lw=0))
        ax.set_title(f"source {s}, image {iid}: {ok.mean():.0%} correct", fontsize=9)
    for ax in axes:
        ax.axis("off")

for s in ranked.index[:N_EXAMPLE_SOURCES]:
    fig, axes = plt.subplots(1, N_EXAMPLE_IMAGES, figsize=(5.2 * N_EXAMPLE_IMAGES, 4))
    show_source(s, axes=np.atleast_1d(axes))
    q = S.loc[s]
    fig.suptitle(f"source {s}: kingdom accuracy {q.acc_kingdom:.2f} (expected {q.expected:.2f}), "
                 f"over-predicts {q.shift_to}", x=0.01, ha="left", fontsize=11)
    fig.tight_layout(); plt.show()

## 8. Diagnosis table

One row per weak source, sorted by kingdom accuracy. Automatic flags (thresholds in `FLAG`):
- `dead coral`: ≥ 20% of scored points are dead coral (labelled animalia, look like substrate).
- `non-turf algae`: ≥ 30% are plantae / chromista (CCA, macroalgae; mostly predicted `none` — the turf convention).
- `rare labels`: ≥ 25% of points in labels with < `RARE_TRAIN` training points.
- `source-specific`: residual ≤ −0.10 (labels that work elsewhere fail here).
- `shift → X`: the predicted kingdom mix is ≥ 0.30 away from the annotated one, towards X.
- `unscored`: ≥ 50% of the source's points have no kingdom label (catch-all / unmapped labels), so its score rests on few points.
- `image outlier`: aspect ratio or resolution outside the training range, or colour statistics far from every training source.

`top error labels`: the three labels with the most kingdom errors, as `label (n): accuracy here / elsewhere → most common prediction`.

In [ ]:
img_far = r.img_dist.quantile(0.9)
def flags(s):
    q = S.loc[s]; f = []
    if q.get("share_dead_coral", 0) >= FLAG["dead_coral"]: f.append("dead coral")
    if q.get("share_other_algae", 0) >= FLAG["other_algae"]: f.append("non-turf algae")
    if q.rare_share >= FLAG["rare"]: f.append("rare labels")
    if q.residual <= FLAG["resid"]: f.append("source-specific")
    if q["shift"] >= FLAG["bias"]: f.append(f"shift → {q.shift_to}")
    if q.unscored >= FLAG["unscored"]: f.append("unscored")
    if (q.aspect > tr_src.aspect.quantile(0.99)) or (q.mpix < tr_src.mpix.quantile(0.05)) or (q.img_dist > img_far):
        f.append("image outlier")
    return ", ".join(f) or "—"

def top_errors(s, n=3):
    x = ls[ls.source_id == s].assign(err=lambda t: t["size"] - t["sum"]).sort_values("err", ascending=False).head(n)
    pred = sc[sc.source_id == s].groupby("label").pred_kingdom.agg(lambda v: v.mode().iloc[0])
    return "; ".join(f"{t.label} ({t['size']}): {t.acc_here:.2f} / " + ("–" if np.isnan(t.acc_else) else f"{t.acc_else:.2f}")
                     + f" → {pred[t.label]}" for _, t in x.iterrows())

diag = S.loc[ranked.loc[WEAK].index, ["points", "acc_kingdom", "expected", "residual", "shift"]].copy()
diag["flags"] = [flags(s) for s in diag.index]
diag["top error labels"] = [top_errors(s) for s in diag.index]
with pd.option_context("display.max_colwidth", 200):
    display(diag.round(2))
print(diag["flags"].str.split(", ").explode().value_counts().to_string())

### Interpretation (cbm_final_res256, all test sources)

**How weak sources were picked.** 139 test sources have at least 200 kingdom-labelled points; the bottom quarter
(kingdom accuracy ≤ 0.68) gives **35 weak sources**. Kingdom ranks sources almost the same way as the mean accuracy
over all six ranks (Spearman 0.91), so these are the weak sources at every rank.

**Two kinds of weakness.** The label mix explains about half of the between-source variance (r = 0.72 between
expected and actual accuracy). The other half is source-specific. Simple global image statistics (brightness,
colour, contrast, sharpness) do **not** explain it (|Spearman| ≤ 0.24). What does is a whole-source **shift**: in
weak sources the model pushes most points towards one kingdom, usually `none` (17 of 35) and sometimes `animalia` (3)
or `plantae` (1). Across sources, shift vs accuracy has Spearman −0.88.

The weak sources fall into four groups (a source can be in more than one):

**A. Concept-table conventions: about half of the weak sources, and the most fixable.** These are the same issues
the rank analysis found, now shown to be what makes whole sources weak.
- **Dead coral** (animalia in the table, looks like substrate, 34% kingdom accuracy, mostly predicted `none`):
  7079, 7080, 3985, 5101, 5102, 7663, 7078, 3357, 2911. Dead coral is about half to two-thirds of the points in the
  worst of these and 79–88% of their errors. 7077–7080 and 5101/5102 look like related surveys (same sizes, 40 images each).
- **Non-turf algae and cyanobacteria predicted `none`**: CCA, Dictyota, Lobophora, Peyssonnelia and cyanobacteria.
  Pooled kingdom accuracy is 0.36 for plantae/chromista and 0.07 for cyanobacteria. Affected sources: 4862 (0.33,
  the weakest source), 4789, 7614, 6026, 2294, 2655, 4292, 4596, 7116. Their labels are just as bad in other sources
  (residual ≈ 0), so this is the turf-is-`none` convention, not these sources.

**B. Unfamiliar communities.** These sources have many points in labels that are rare or absent in training, plus
organisms the model barely saw.
- **4254**: 59% rare labels and 28% absent (Champia, Zonaria, Hormosira). The labels suggest a temperate
  algal reef.
- **3765**: a fouling-type community of colonial ascidians, bryozoans, polychaetes and red algae (41% rare labels,
  81% of points in labels found in no other test source).
- **7614**: Ecklonia kelp and sea urchins. The labels again suggest a temperate reef.
- **6673**: Colpomenia, Halimeda discoidea and Caulerpa.
- **800**: ARMS settlement plates rather than reef photos (`arms-crep-*` labels, 86% unique to this source). Its CCA
  and encrusting algae are read as animals. This is the second-largest test source (127k points), so it noticeably
  drags down the pooled numbers.

**C. Image format and resolution.** Labels that work elsewhere fail here.
- **1588**: panoramic transect strips. The median aspect ratio is 2.5; only about 4 of 682 training sources have a median
  aspect ratio above 2, and 99% are ≤ 1.78. The images are squashed into 256×256 with about 330 points each. CCA goes to `none` 94% of the time
  and corals about 50%.
- **4584**: 0.2 MP images, against 1.2 MP for the 5th percentile of training sources. Porites compressa is called
  `none` 31% of the time here vs 7% elsewhere.
- **7358**: an extreme green/blue colour cast (mean saturation 0.92). Hydrozoans are called `none` 92% of the time.
  The training ColorJitter (saturation ±0.1, hue ±0.03) doesn't come close to this.
- **Resolution-limited detail**: the 768 px model fixes much of 4692 (+0.33 to 0.91: rock read as animals at
  256 px) and 6673 (+0.25). It helps 1832, 2616, 7116 and 546 by 0.12–0.15. It barely changes groups A and B.
  Caveat: the 768 model uses the taxv2 table, so a few labels have different ground truth (for example `all other`
  in 546, and probably 3985's −0.19).

**D. Source-specific annotation conventions (needs a manual look).**
- **2616** (47k points): dark, heavily shadowed images. `pavement`, `shadow`, `turf algae` and
  `sediment covered limestone` are predicted `animalia` 57–78% of the time. Elsewhere the same labels are 56–83%
  correct.
- **3295** (21 images): sand and bare substrate are predicted plantae 71–75% of the time, against 96–97% correct
  elsewhere. Corals are called `none`. The points span only the top-left ~79% of each image, so I tested a
  coordinate-scale mismatch by re-scoring with coordinates ×1.117. Accuracy stayed at 0.33, so the points are not
  misaligned. In the example images, "bare substrate" points sit on algae-covered rock, so this is most likely an
  annotation convention.
- **546**: the catch-all label `all other` (1,216 points) is kingdom `none` but predicted animalia 80% of the time.
  The v2 concept table already changes it to `not_given`.
- **1832**: `porites compressa algal overgrowth`, a label specific to this source, is predicted `none`/plantae.
  88% of 1832's points have no kingdom label, so its score rests on 203 points.

**What would move the weak sources:**
1. The concept-table changes (dead coral, turf → `not_given`, image-quality labels). These target group A, about
   half the weak sources. They can be checked without retraining by re-scoring these dumps with the revised ground
   truth.
2. Inference that keeps aspect ratio and detail: tiling or sliding windows for large or panoramic images
   (1588, 4692, 6673, 2616).
3. Training augmentation for colour casts and low resolution (7358, 4584).
4. Reporting ARMS plates (800) and the non-tropical or fouling sources (4254, 3765, 7614) separately, or adding
   similar sources to training. They are a different task from tropical reef photo-quadrats.
5. A manual review of 2616 and 3295 annotations.